In [7]:
import os
import json
import asyncio
import re
import random
import hashlib
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
import requests
from temporalio import activity
from mistralai import Mistral
from dotenv import load_dotenv
import time

# --- ASTUCE POUR LE CHEMIN DES CLÉS ---
# --- CHARGEMENT DIRECT DES CLÉS ---
# Puisque tu es à la racine, on pointe directement sur le fichier
env_path = Path("cle.env").absolute()
load_dotenv(dotenv_path=env_path, override=True)

if not os.getenv("MISTRAL_API_KEY"):
    print(f"❌ Erreur : Impossible de trouver les clés dans {env_path}")
else:
    print(f"✅ Clés chargées depuis {env_path}")

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

# --- CONSTANTES & CONFIGURATION ---
SOCIAL_BLACKLIST = ["tiktok.com", "facebook.com", "instagram.com", "x.com", "twitter.com"]

# 🏆 NOUVEAU : Hiérarchie stricte des sources
SOURCES_TIER_1 = ["insee.fr", "gouv.fr", "ameli.fr", "vie-publique.fr", "senat.fr", "assemblee-nationale.fr", "ccomptes.fr", "service-public.fr"]
SOURCES_TIER_2 = ["lemonde.fr", "lefigaro.fr", "liberation.fr", "lesechos.fr", "afp.com", "francetvinfo.fr", "radiofrance.fr"]

# 🧠 NOUVEAU : Cache en mémoire pour garantir 100% de constance
CACHE_REQUETES = {}

DEFAULT_FACT_CHECK_POST_URL = "http://localhost:8000/api/stream/fact-check"
MISTRAL_WEB_SEARCH_MODEL = os.getenv("MISTRAL_WEB_SEARCH_MODEL", "mistral-medium-latest").strip()
SOURCE_SELECTION_MODE = os.getenv("SOURCE_SELECTION_MODE", "heuristic").strip().lower()
# On augmente la patience du script pour qu'il attende que Mistral le débloque
MISTRAL_RATE_LIMIT_MAX_RETRIES = int(os.getenv("MISTRAL_RATE_LIMIT_MAX_RETRIES", "6")) # 6 essais au lieu de 4
MISTRAL_RATE_LIMIT_BACKOFF_MAX_SECONDS = float(os.getenv("MISTRAL_RATE_LIMIT_BACKOFF_MAX_SECONDS", "20.0")) # Attend jusqu'à 20s au lieu de 6s
MISTRAL_RATE_LIMIT_BACKOFF_BASE_SECONDS = float(os.getenv("MISTRAL_RATE_LIMIT_BACKOFF_BASE_SECONDS", "0.7"))

CORRECTION_MARKERS = ["pardon", "je corrige", "je me corrige", "je me suis trompe", "je me suis trompé", "plutot", "plutôt", "rectification", "en fait", "non"]
FRENCH_STOPWORDS = {"alors", "avec", "avoir", "bien", "cette", "dans", "dont", "elle", "elles", "entre", "etre", "fait", "faire", "mais", "meme", "nous", "pour", "plus", "pas", "que", "qui", "sans", "sont", "sur", "tout", "tous", "tres", "une", "des", "les", "du", "de", "la", "le", "un", "est", "et", "ou", "en", "il", "ils", "elle", "on", "je", "tu", "vous", "nous"}

# --- UTILITAIRES ---

def _is_rate_limited_error(exc: Exception) -> bool:
    lowered = str(exc).lower()
    return "rate limit" in lowered or "status 429" in lowered or "\"code\":\"1300\"" in lowered or "'code':'1300'" in lowered

async def _mistral_json(prompt: str, *, model: str = "mistral-small-latest") -> dict[str, Any]:
    max_retries = max(0, MISTRAL_RATE_LIMIT_MAX_RETRIES)
    for attempt in range(max_retries + 1):
        try:
            res = await client.chat.complete_async(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.0,  # 🔒 NOUVEAU : Zéro créativité
                random_seed=42    # 🔒 NOUVEAU : Constance absolue
            )
            parsed = json.loads(res.choices[0].message.content)
            if isinstance(parsed, dict):
                return parsed
            return {"value": parsed}
        except Exception as exc:
            should_retry = _is_rate_limited_error(exc) and attempt < max_retries
            if not should_retry: raise
            backoff = min(MISTRAL_RATE_LIMIT_BACKOFF_MAX_SECONDS, MISTRAL_RATE_LIMIT_BACKOFF_BASE_SECONDS * (2 ** attempt))
            jitter = random.uniform(0.0, 0.25 * max(0.2, backoff))
            wait_seconds = backoff + jitter
            print(f"[mistral] rate limit detecte, retry {attempt + 1}/{max_retries} dans {wait_seconds:.2f}s")
            await asyncio.sleep(wait_seconds)

def _tokenize(text: str) -> list[str]:
    tokens = re.findall(r"[a-zA-ZÀ-ÿ0-9]+", (text or "").lower())
    return [t for t in tokens if len(t) >= 3 and t not in FRENCH_STOPWORDS]

def _extract_numbers(text: str) -> list[str]:
    if not isinstance(text, str): return []
    return re.findall(r"\d+(?:[.,]\d+)?", text)

def _extract_current_affirmation(current_json: dict) -> str:
    current = current_json.get("affirmation_courante")
    if isinstance(current, str) and current.strip(): return current.strip()
    fallback = current_json.get("affirmation")
    if isinstance(fallback, str) and fallback.strip(): return fallback.strip()
    return ""

def _extract_previous_context_phrases(last_minute_json: dict, current_affirmation: str) -> list[str]:
    previous_phrases = last_minute_json.get("previous_phrases")
    if isinstance(previous_phrases, list):
        return [p.strip() for p in previous_phrases if isinstance(p, str) and p.strip()]
    phrases = [p.strip() for p in last_minute_json.get("phrases", []) if isinstance(p, str) and p.strip()]
    if phrases and current_affirmation and phrases[-1] == current_affirmation:
        return phrases[:-1]
    return phrases

def _is_valid_http_url(url: str) -> bool:
    if not isinstance(url, str): return False
    stripped = url.strip().lower()
    return stripped.startswith("http://") or stripped.startswith("https://")

def _domain_to_organization(url: str) -> str:
    try: host = urlparse(url).netloc.lower()
    except Exception: host = ""
    if host.startswith("www."): host = host[4:]
    return host or "source-inconnue"

# 🎯 NOUVEAU : Fonction de Scoring des Sources
def _score_source_url(url: str) -> float:
    url_lower = url.lower()
    if any(domain in url_lower for domain in SOURCES_TIER_1): return 1000.0
    if any(domain in url_lower for domain in SOURCES_TIER_2): return 500.0
    return 10.0

# 🔒 NOUVEAU : Générateur de recherche déterministe avec Cache
def generer_requete_deterministe(affirmation: str, prefix: str) -> str:
    phrase_id = hashlib.md5(affirmation.strip().lower().encode()).hexdigest()
    cache_key = f"{prefix}_{phrase_id}"
    
    if cache_key in CACHE_REQUETES:
        print(f"⚡ [CACHE HIT] Requête réutilisée.")
        return CACHE_REQUETES[cache_key]
        
    mots_cles = " ".join(_tokenize(affirmation))
    requete_stricte = f"{prefix} {mots_cles} France"
    CACHE_REQUETES[cache_key] = requete_stricte
    return requete_stricte

def _heuristic_self_correction(current_affirmation: str, next_affirmation: str) -> dict[str, Any]:
    normalized_next = (next_affirmation or "").strip().lower()
    if not normalized_next:
        return {"next_is_correction": False, "confidence": 0.0, "reason": "", "detector": "heuristic"}

    words = re.findall(r"[a-zA-ZÀ-ÿ0-9]+", normalized_next)
    has_marker = any(marker in normalized_next for marker in CORRECTION_MARKERS)
    current_numbers = _extract_numbers(current_affirmation)
    next_numbers = _extract_numbers(next_affirmation)
    replaces_number = bool(current_numbers and next_numbers and current_numbers != next_numbers)
    short_followup = len(words) <= 10

    if has_marker and (replaces_number or short_followup):
        return {"next_is_correction": True, "confidence": 0.95 if replaces_number else 0.85, "reason": "Correction explicite", "detector": "heuristic"}
    if replaces_number and short_followup:
        return {"next_is_correction": True, "confidence": 0.75, "reason": "Valeur numérique remplacée", "detector": "heuristic"}

    return {"next_is_correction": False, "confidence": 0.2, "reason": "Pas de signal", "detector": "heuristic"}

def _fallback_select_relevant_sources(*, assertion: str, question: str, candidates: list[dict[str, Any]]) -> list[int]:
    needed_tokens = set(_tokenize(f"{assertion} {question}"))
    if not candidates: return []
    if not needed_tokens: return [int(candidates[0]["id"])]

    scored: list[tuple[float, int]] = []
    for candidate in candidates:
        haystack = " ".join([str(candidate.get("title", "")), str(candidate.get("snippet", "")), str(candidate.get("url", ""))])
        candidate_tokens = set(_tokenize(haystack))
        overlap = len(needed_tokens.intersection(candidate_tokens))
        if overlap > 0:
            # 🎯 NOUVEAU : Application du multiplicateur d'autorité
            base_score = overlap / max(1, len(needed_tokens))
            authority_bonus = _score_source_url(str(candidate.get("url", "")))
            score = base_score * authority_bonus
            scored.append((score, int(candidate["id"])))
            
    scored.sort(reverse=True)
    return [source_id for _, source_id in scored[:3]]

async def _mistral_web_search_response(query: str) -> Any:
    max_retries = max(0, MISTRAL_RATE_LIMIT_MAX_RETRIES)
    for attempt in range(max_retries + 1):
        try:
            return await client.beta.conversations.start_async(
                model=MISTRAL_WEB_SEARCH_MODEL,
                inputs=query,
                tools=[{"type": "web_search"}],
                completion_args={"temperature": 0.0, "random_seed": 42} # 🔒 Zéro créativité sur le search aussi
            )
        except Exception as exc:
            should_retry = _is_rate_limited_error(exc) and attempt < max_retries
            if not should_retry: raise
            backoff = min(MISTRAL_RATE_LIMIT_BACKOFF_MAX_SECONDS, MISTRAL_RATE_LIMIT_BACKOFF_BASE_SECONDS * (2 ** attempt))
            jitter = random.uniform(0.0, 0.25 * max(0.2, backoff))
            wait_seconds = backoff + jitter
            print(f"[mistral-web-search] rate limit detecte, retry {attempt + 1}/{max_retries} dans {wait_seconds:.2f}s")
            await asyncio.sleep(wait_seconds)

def _collect_candidates_from_any(value: Any, collected: list[dict[str, str]]) -> None:
    if isinstance(value, dict):
        raw_url = value.get("url")
        if isinstance(raw_url, str) and _is_valid_http_url(raw_url):
            title = str(value.get("title", "")).strip()
            snippet = str(value.get("description", "") or value.get("snippet", "") or value.get("content", "")).strip()
            collected.append({"url": raw_url.strip(), "title": title, "snippet": snippet[:480]})
        for inner in value.values(): _collect_candidates_from_any(inner, collected)
        return
    if isinstance(value, list):
        for item in value: _collect_candidates_from_any(item, collected)

def _extract_mistral_web_candidates(response: Any) -> list[dict[str, str]]:
    payload = response.model_dump() if hasattr(response, "model_dump") else response
    if not isinstance(payload, dict): return []
    outputs = payload.get("outputs", [])
    if not isinstance(outputs, list): return []

    candidates: list[dict[str, str]] = []
    for output in outputs:
        if not isinstance(output, dict): continue
        output_type = str(output.get("type", ""))
        if output_type == "message.output": _collect_candidates_from_any(output.get("content"), candidates)
        elif output_type == "tool.execution": _collect_candidates_from_any(output.get("info"), candidates)
        
    deduped: list[dict[str, str]] = []
    seen: set[str] = set()
    for candidate in candidates:
        url = str(candidate.get("url", "")).strip()
        if not _is_valid_http_url(url) or url in seen: continue
        try: host = urlparse(url).netloc.lower()
        except Exception: host = ""
        if host.startswith("www."): host = host[4:]
        if any(host == blocked or host.endswith(f".{blocked}") for blocked in SOCIAL_BLACKLIST): continue
        seen.add(url)
        deduped.append({"url": url, "title": str(candidate.get("title", "")).strip(), "snippet": str(candidate.get("snippet", "")).strip()[:480]})
    return deduped

async def _search_relevant_sources(*, assertion: str, question: str, query: str, search_depth: str = "basic", max_results: int = 6, exclude_domains: list[str] | None = None) -> list[dict[str, str]]:
    excluded = set(exclude_domains or [])
    excluded.update(SOCIAL_BLACKLIST)
    web_query = f"{query}\n\nTrouve des pages pertinentes qui répondent directement à la vérification."
    
    try:
        response = await _mistral_web_search_response(web_query)
        raw_candidates = _extract_mistral_web_candidates(response)
    except Exception as exc:
        print(f"❌ Erreur Mistral Web Search: {exc}")
        raw_candidates = []

    filtered_candidates: list[dict[str, str]] = []
    for item in raw_candidates:
        url = str(item.get("url", "")).strip()
        if not _is_valid_http_url(url): continue
        try: host = urlparse(url).netloc.lower()
        except Exception: host = ""
        if host.startswith("www."): host = host[4:]
        if any(host == blocked or host.endswith(f".{blocked}") for blocked in excluded): continue
        filtered_candidates.append(item)
        if len(filtered_candidates) >= max_results: break

    candidates: list[dict[str, Any]] = [{"id": idx, "url": item["url"], "title": str(item.get("title", "")).strip(), "snippet": str(item.get("snippet", "")).strip()[:480]} for idx, item in enumerate(filtered_candidates, start=1)]
    if not candidates: return []

    selected_ids = _fallback_select_relevant_sources(assertion=assertion, question=question, candidates=candidates)
    selected_ids = list(dict.fromkeys(selected_ids))[:3]
    return [{"organization": _domain_to_organization(item["url"]), "url": item["url"], "title": item["title"], "snippet": item["snippet"]} for item in candidates if int(item["id"]) in selected_ids]

def _sources_for_prompt(sources: list[dict[str, str]]) -> str:
    if not sources: return "Aucune source."
    return "\n".join([f"[{idx}] org={s.get('organization', '')} url={s.get('url', '')} snippet={s.get('snippet', '')}" for idx, s in enumerate(sources, start=1)])

def _sanitize_primary_source(*, raw_result: dict[str, Any], sources: list[dict[str, str]]) -> dict[str, Any]:
    sanitized = dict(raw_result)
    if not sources:
        sanitized.update({"source_is_relevant": False, "nom_source": "", "url_source": "", "sources": []})
        return sanitized

    preferred_source = sources[0]
    try: selected_index = int(raw_result.get("source_index", 1))
    except (TypeError, ValueError): selected_index = 1
    if 1 <= selected_index <= len(sources): preferred_source = sources[selected_index - 1]

    sanitized.update({
        "source_is_relevant": True,
        "nom_source": preferred_source.get("organization", ""),
        "url_source": preferred_source.get("url", ""),
        "sources": [{"organization": s.get("organization", ""), "url": s.get("url", "")} for s in sources if _is_valid_http_url(s.get("url", ""))]
    })
    return sanitized

def _collect_sources_from_reports(rapports_agents: list[dict[str, Any]]) -> list[dict[str, str]]:
    seen, collected = set(), []
    for report in rapports_agents:
        if not isinstance(report, dict): continue
        for source in report.get("sources", []):
            if not isinstance(source, dict): continue
            url = str(source.get("url", "")).strip()
            if not _is_valid_http_url(url) or url in seen: continue
            seen.add(url)
            collected.append({"organization": str(source.get("organization", "")).strip() or _domain_to_organization(url), "url": url})
        single_url = str(report.get("url_source", "")).strip()
        if _is_valid_http_url(single_url) and single_url not in seen:
            seen.add(single_url)
            collected.append({"organization": str(report.get("nom_source", "")).strip() or _domain_to_organization(single_url), "url": single_url})
    return collected

def _first_source_for_agent(rapports_agents: list[dict[str, Any]], agent_name: str) -> dict[str, str] | None:
    for report in rapports_agents:
        if not isinstance(report, dict) or str(report.get("agent", "")).strip() != agent_name: continue
        for source in report.get("sources", []):
            if isinstance(source, dict) and _is_valid_http_url(str(source.get("url", "")).strip()):
                url = str(source.get("url", "")).strip()
                return {"organization": str(source.get("organization", "")).strip() or _domain_to_organization(url), "url": url}
        url = str(report.get("url_source", "")).strip()
        if _is_valid_http_url(url):
            return {"organization": str(report.get("nom_source", "")).strip() or _domain_to_organization(url), "url": url}
    return None

def _enrich_editor_result_with_sources(result: dict[str, Any], rapports_agents: list[dict[str, Any]]) -> dict[str, Any]:
    enriched = dict(result)
    enriched["sources"] = _collect_sources_from_reports(rapports_agents)
    explications = enriched.get("explications")
    
    if isinstance(explications, dict):
        mapping = {"statistique": "statistique", "contexte": "contexte", "coherence": "coherence"}
        for explication_key, agent_name in mapping.items():
            source = _first_source_for_agent(rapports_agents, agent_name)
            explication_value = explications.get(explication_key)
            if not source:
                if isinstance(explication_value, dict):
                    explication_value.pop("source", None)
                    explication_value.pop("url", None)
                continue
            if isinstance(explication_value, dict):
                explication_value.setdefault("source", source["organization"])
                explication_value["url"] = source["url"]
            elif isinstance(explication_value, str) and explication_value.strip():
                explications[explication_key] = {"texte": explication_value.strip(), "source": source["organization"], "url": source["url"]}
    return enriched

# --- AGENTS EXPERTS ---

async def agent_statistique(data):
    print(f"📊 [Agent Stat] Investigation approfondie en cours...")
    
    # 🔒 NOUVEAU : Requête nettoyée et mise en cache
    query_complete = generer_requete_deterministe(data['affirmation'], "statistiques officielles chiffres exacts")
    
    selected_sources = await _search_relevant_sources(
        assertion=data.get("affirmation", ""), question=data.get("question_posee", ""),
        query=query_complete, search_depth="basic", max_results=8
    )
    if not selected_sources:
        return {"agent": "statistique", "verdict": "indetermine", "analyse_detaillee": "Aucune source suffisamment pertinente trouvée.", "source_is_relevant": False, "nom_source": "", "url_source": "", "sources": []}

    prompt = f"""Vérifie cette affirmation : '{data['affirmation']}'.
    Sources validées (utilise UNIQUEMENT celles-ci) :
    {_sources_for_prompt(selected_sources)}

    MISSION DE CONSENSUS (RÈGLES D'OR) :
    1. Compare les chiffres présents dans les différentes sources.
    2. En cas de contradiction, la règle est absolue : l'INSEE et les sites en '.gouv.fr' ont TOUJOURS raison.
    3. Ton verdict final DOIT s'appuyer sur la source la plus haute dans la hiérarchie officielle.
    4. verdict = "vrai", "faux", "exagéré", "trompeur".
    5. source_index = index numérique de la source la plus fiable utilisée (1..N).

    JSON: {{"agent": "statistique", "verdict": "vrai|faux|...", "analyse_detaillee": "...", "source_index": 1}}"""
    
    raw = await _mistral_json(prompt)
    return _sanitize_primary_source(raw_result=raw, sources=selected_sources)

async def agent_rhetorique(data):
    print(f"🧠 [Agent Rhétorique] Analyse logique...")
    prompt = f"""Analyse : Question posée : '{data.get('question_posee', '')}' | Réponse : '{data.get('affirmation', '')}'
    RÈGLES STRICTES :
    1. Si la personne répond à la question : Laisse "explication" VIDE "".
    2. Si la personne esquive : Explique en une courte phrase.
    JSON: {{"agent": "rhetorique", "explication": "..."}}"""
    return await _mistral_json(prompt)

async def agent_coherence_personnelle(data):
    print(f"🕵️ [Agent Cohérence] Accès archives pour {data['personne']}...")
    # Requête ciblée sur les anciennes déclarations
    query_complete = generer_requete_deterministe(f"citation ancienne déclaration {data['personne']} {data['affirmation']}", "archives")
    selected_sources = await _search_relevant_sources(assertion=data.get("affirmation", ""), question=data.get("question_posee", ""), query=query_complete, search_depth="advanced", max_results=5)
    
    if not selected_sources: 
        return {"agent": "coherence", "explication": "", "source_is_relevant": False, "sources": []}

    prompt = f"""Tu es un expert en archives politiques.
    Tu vérifies UNIQUEMENT si {data['personne']} a tenu des propos contradictoires dans le PASSÉ concernant : '{data['affirmation']}'.
    
    Sources validées : {_sources_for_prompt(selected_sources)}

    RÈGLES STRICTES :
    1. Ton SEUL but est de trouver une ancienne déclaration (citation) de cette personne.
    2. INTERDICTION de vérifier si le fait est vrai ou faux. Laisse ça à l'agent statistique.
    3. Si tu ne trouves aucune citation passée de {data['personne']} qui contredit sa phrase actuelle, laisse "explication" VIDE "".
    4. Si tu trouves une contradiction, rédige: "{data['personne']} avait affirmé que [ancienne citation]".

    JSON: {{"agent": "coherence", "explication": "...", "source_index": 1}}"""
    
    raw = await _mistral_json(prompt)
    return _sanitize_primary_source(raw_result=raw, sources=selected_sources)

async def agent_contexte(data):
    print(f"📚 [Agent Contexte] Analyse factuelle détaillée...")
    query_complete = generer_requete_deterministe(data['affirmation'], "faits réels historique contexte")
    selected_sources = await _search_relevant_sources(assertion=data.get("affirmation", ""), question=data.get("question_posee", ""), query=query_complete, search_depth="basic", max_results=8)
    
    if not selected_sources: return {"agent": "contexte", "analyse_detaillee": "Aucun contexte trouvé.", "source_is_relevant": False, "sources": []}

    prompt = f"""Analyse le contexte de : '{data['affirmation']}'.
    Sources : {_sources_for_prompt(selected_sources)}
    Si les sources officielles contredisent l'affirmation, explique brièvement la vérité.
    JSON: {{"agent": "contexte", "analyse_detaillee": "...", "source_index": 1}}"""
    raw = await _mistral_json(prompt)
    return _sanitize_primary_source(raw_result=raw, sources=selected_sources)

async def agent_routeur(data):
    prompt = f"""Tu es un routeur logique strict (un arbre de décision IF/ELSE).
    
    QUESTION POSÉE À L'ORATEUR : '{data.get('question_posee', '')}'
    AFFIRMATION ACTUELLE : '{data.get('affirmation', '')}'

    RÈGLE DE NETTOYAGE :
    - Garde uniquement l'affirmation finale (efface les bafouillements).
    - Garde uniquement l'affirmation vérifiable (efface les goûts personnels et les salutations)

    ARBRE DE DÉCISION (UN SEUL AGENT DOIT ÊTRE À TRUE) :
    Évalue ces conditions dans l'ordre exact (1 à 4) et arrête-toi à la première qui correspond :

    1. STATISTIQUE : Y a-t-il un chiffre, un pourcentage, ou un mot lié à une quantité ("aucun", "plus aucun", "un peu", "moins", "zéro", "tous", "moitié") dans l'affirmation ?
       -> Si OUI : "run_stats": true (et tous les autres à false).
    
    2. CONTEXTE : Y a-t-il un événement précis mentionné (Jeux Olympiques, guerre, attentats, crise, loi) ?
       -> Si OUI : "run_contexte": true (et tous les autres à false).
       
    3. RHÉTORIQUE : Une question a-t-elle été posée à l'orateur (le champ QUESTION POSÉE n'est pas vide) ?
       -> Si OUI : "run_rhetorique": true (et tous les autres à false).
       
    4. COHÉRENCE : S'il n'y a PAS de chiffre, PAS d'événement, et PAS de question.
       -> Alors : "run_coherence_personnelle": true (et tous les autres à false).

    JSON STRICT ATTENDU :
    {{
      "affirmation_propre": "La phrase nettoyée",
      "run_stats": false,
      "run_rhetorique": false,
      "run_coherence_personnelle": false,
      "run_contexte": false
    }}
    """
    return await _mistral_json(prompt)

async def executer_analyse_parallele(data):
    print(f"🚦 PHRASE ACTUELLE : '{data['affirmation'][:60]}...'")
    routage = await agent_routeur(data)
    affirmation_propre = routage.get('affirmation_propre', data['affirmation'])
    print(f"✨ TEXTE NETTOYÉ : '{affirmation_propre}'")
    
    data_propre = data.copy()
    data_propre['affirmation'] = affirmation_propre 
    
    taches_a_lancer = []
    if routage.get("run_stats"): taches_a_lancer.append(agent_statistique(data_propre))
    if routage.get("run_rhetorique"): taches_a_lancer.append(agent_rhetorique(data_propre))
    if routage.get("run_coherence_personnelle"): taches_a_lancer.append(agent_coherence_personnelle(data_propre))
    if routage.get("run_contexte"): taches_a_lancer.append(agent_contexte(data_propre))
        
    if not taches_a_lancer: return [] 
    
    print(f"🚀 EXÉCUTION : {len(taches_a_lancer)} agent(s) en parallèle...")
    return await asyncio.gather(*taches_a_lancer)

# --- 🎯 NOUVEAU : LE RÉDACTEUR EN CHEF DICTATORIAL ---
async def agent_editeur(contexte_precedent, affirmation_actuelle, rapports_agents):
    print("📝 [Rédacteur en Chef] Formatage strict pour affichage TV...")
    if not rapports_agents:
        return {"afficher_bandeau": False, "raison": "Aucun fact-check nécessaire."}

    prompt = f"""Tu es le switch logique final d'un logiciel de régie TV.
    
    AFFIRMATION : "{affirmation_actuelle}"
    RAPPORTS : {json.dumps(rapports_agents, ensure_ascii=False)}

    RÈGLE DE FORMATAGE ABSOLUE POUR "texte" (1 seule phrase, AUCUNE EXCEPTION) :
    - CAS STATISTIQUE FAUSSE : Tu DOIS écrire "FAUX : La réalité est de [Le Vrai Chiffre]."
    - CAS STATISTIQUE EXAGÉRÉE : Tu DOIS écrire "EXAGÉRÉ : Le chiffre réel est [Le Vrai Chiffre]."
    - CAS CONTEXTE TROMPEUR : Tu DOIS écrire "TROMPEUR : [Une phrase très courte qui rétablit le fait]."
    - CAS ESQUIVE : Tu DOIS écrire "ESQUIVE : L'invité ne répond pas sur [Le sujet de la question]."
    - CAS VRAI / INCERTAIN / MANQUE DE PREUVE : Laisse "afficher_bandeau" à false.

    INTERDICTION : Ne fais aucune phrase d'introduction. Ne mets AUCUNE source dans le champ "texte" (elles iront ailleurs). Sois concis et brut.
    
    FORMAT JSON STRICT ATTENDU :
    {{
      "afficher_bandeau": true,
      "verdict_global": "Faux",
      "explications": {{
         "statistique": {{
            "texte": "FAUX : La réalité est de 5,5 %."
         }},
         "contexte": {{
            "texte": "TROMPEUR : La loi a été rejetée le mois dernier."
         }}
      }}
    }}
    NOTE : N'inclus dans 'explications' que l'agent pertinent. Laisse les champs URL/Source vides dans le prompt (le code Python s'en charge).
    """
    return await _mistral_json(prompt)

@activity.defn
async def analyze_debate_line(current_json: dict, last_minute_json: dict) -> dict:
    personne = current_json.get("personne", "Intervenant inconnu")
    question = current_json.get("question_posee", "")
    affirmation = _extract_current_affirmation(current_json)
    if not affirmation: return {"afficher_bandeau": False, "raison": "Phrase courante vide."}
    
    phrases_contexte = _extract_previous_context_phrases(last_minute_json, affirmation)
    contexte_precedent = " ".join(phrases_contexte)

    data_pour_agents = {"personne": personne, "question_posee": question, "affirmation": affirmation, "contexte_precedent": contexte_precedent}

    try:
        rapports_bruts = await executer_analyse_parallele(data_pour_agents)
        resultat_final = await agent_editeur(contexte_precedent, affirmation, rapports_bruts)
        
        if not isinstance(resultat_final, dict): return {"afficher_bandeau": False, "raison": "Resultat editeur invalide."}
        
        resultat_final = _enrich_editor_result_with_sources(resultat_final, rapports_bruts)
        
        # 🔒 Sécurité Fallback : On nettoie si Mistral a été trop bavard sur le champ texte
        if resultat_final.get("afficher_bandeau"):
            for exp in resultat_final.get("explications", {}).values():
                if isinstance(exp, dict) and "texte" in exp:
                    texte = exp["texte"]
                    # On coupe à la première phrase si ce n'est pas déjà le cas
                    if "." in texte and len(texte) > 80:
                        exp["texte"] = texte.split(".")[0] + "."

        if resultat_final.get("afficher_bandeau") and not resultat_final.get("sources"):
            resultat_final["afficher_bandeau"] = False
            resultat_final["raison"] = "Fact-check ignoré: aucune source pertinente avec lien URL."
            
        return resultat_final

    except Exception as e:
        print(f"❌ Erreur lors de l'analyse : {e}")
        return {"afficher_bandeau": False, "erreur": str(e)}

@activity.defn
async def check_next_phrase_self_correction(
    current_json: dict, next_json: dict | None, last_minute_json: dict
) -> dict:
    """
    Détecte si la phrase suivante corrige / annule la phrase courante.
    """
    current_affirmation = _extract_current_affirmation(current_json)
    next_affirmation = _extract_current_affirmation(next_json or {})

    if not current_affirmation:
        return {
            "has_next_phrase": bool(next_affirmation),
            "next_is_correction": False,
            "confidence": 0.0,
            "reason": "Phrase courante vide.",
        }

    if not next_affirmation:
        return {
            "has_next_phrase": False,
            "next_is_correction": False,
            "confidence": 0.0,
            "reason": "Aucune phrase suivante complete.",
        }

    # 1. Vérification rapide et "gratuite" (sans IA)
    heuristic = _heuristic_self_correction(current_affirmation, next_affirmation)
    if bool(heuristic.get("next_is_correction")):
        return {
            "has_next_phrase": True,
            "next_is_correction": True,
            "confidence": float(heuristic.get("confidence", 0.0)),
            "reason": str(heuristic.get("reason", "")),
            "detector": "heuristic",
            "current_affirmation": current_affirmation,
            "next_affirmation": next_affirmation,
        }

    # 2. Vérification approfondie avec le LLM si l'heuristique n'est pas sûre
    contexte_precedent = " ".join(
        _extract_previous_context_phrases(last_minute_json, current_affirmation)
    )
    prompt = f"""Tu es un detecteur de correction en direct.

PHRASE_COURANTE:
"{current_affirmation}"

PHRASE_SUIVANTE:
"{next_affirmation}"

CONTEXTE_PRECEDENT:
"{contexte_precedent}"

Mission:
- Determine si PHRASE_SUIVANTE corrige/retracte explicitement PHRASE_COURANTE.
- Exemples de correction: "je corrige", "je me suis trompe", "non plutot", nouveau chiffre qui remplace le precedent.
- Si PHRASE_SUIVANTE est seulement un ajout, une nouvelle idee ou une reformulation, alors ce n'est PAS une correction.

Renvoie UNIQUEMENT ce JSON strict:
{{
  "next_is_correction": true|false,
  "confidence": 0.0,
  "reason": "court"
}}
"""
    try:
        parsed = await _mistral_json(prompt)
        next_is_correction = bool(parsed.get("next_is_correction", False))
        try:
            confidence = float(parsed.get("confidence", 0.0))
        except (TypeError, ValueError):
            confidence = 0.0
        confidence = max(0.0, min(1.0, confidence))
        reason = str(parsed.get("reason", "")).strip()
        return {
            "has_next_phrase": True,
            "next_is_correction": next_is_correction,
            "confidence": confidence,
            "reason": reason,
            "detector": "llm",
            "current_affirmation": current_affirmation,
            "next_affirmation": next_affirmation,
        }
    except Exception as exc:
        # LLM indisponible (ex: 429 rate limit) -> fallback heuristique déjà calculé.
        return {
            "has_next_phrase": True,
            "next_is_correction": bool(heuristic.get("next_is_correction", False)),
            "confidence": float(heuristic.get("confidence", 0.0)),
            "reason": (
                f"LLM indisponible, fallback heuristique: "
                f"{heuristic.get('reason', 'indetermine')}"
            ),
            "detector": "heuristic_fallback_after_llm_error",
            "error": str(exc),
            "current_affirmation": current_affirmation,
            "next_affirmation": next_affirmation,
        }
@activity.defn
async def post_fact_check_result(payload: dict) -> dict:
    url = os.getenv("FACT_CHECK_POST_URL", DEFAULT_FACT_CHECK_POST_URL)
    def _do_post(): return requests.post(url, json=payload, timeout=10)
    try:
        response = await asyncio.to_thread(_do_post)
        return {"posted": response.ok, "status_code": response.status_code, "url": url, "response_body_preview": response.text[:1000]}
    except Exception as exc:
        return {"posted": False, "url": url, "error": str(exc)}

✅ Clés chargées depuis C:\Users\emmag\hackathon-paris\cle.env


In [8]:
# --- TEST DU NOUVEAU WORKFLOW ---
async def tester_pipeline_tv():
    # Simulation des données de contexte
    contexte_historique = "Je pense qu'il y a 30% de SDF à Paris. Il y a 90% de base en France."
    
    # Données à l'instant T
    data_instant_t = {
        "personne": "Gabriel Attal",
        "question_posee": "",
        "affirmation": "La dette de la sécurité sociale est de 30 milliards. J'aime le chocolat et les beignets",
        "contexte_precedent": contexte_historique # Ajouté pour le nouveau Routeur
    }

    start_time = time.perf_counter()
    print(f"🎬 DÉBUT DU TEST POUR : {data_instant_t['personne']}")
    
    # 1. Analyse brute par les agents (renvoie uniquement la liste des rapports)
    rapports_bruts = await executer_analyse_parallele(data_instant_t)
    
    # 2. Arbitrage par le Juge (Rédacteur en chef)
    bandeau_tv = await agent_editeur(
        contexte_precedent=contexte_historique, 
        affirmation_actuelle=data_instant_t["affirmation"], 
        rapports_agents=rapports_bruts
    )
    
    # 3. 🎯 NOUVEAU : Enrichissement des sources
    # C'est cette fonction qui va "coller" les URL INSEE/Gouv.fr trouvées par les experts
    # dans le JSON final du Juge pour l'affichage OBS.
    if isinstance(bandeau_tv, dict):
        bandeau_tv_final = _enrich_editor_result_with_sources(bandeau_tv, rapports_bruts)
    else:
        bandeau_tv_final = {"erreur": "Le Juge n'a pas renvoyé un dictionnaire valide."}
    
    end_time = time.perf_counter()
    
    print(f"\n⏱️ Temps total du Pipeline : {end_time - start_time:.2f}s")
    print("\n--- BANDEAU OBS FINAL ---")
    print(json.dumps(bandeau_tv_final, indent=2, ensure_ascii=False))

# Lancement du test
await tester_pipeline_tv()


🎬 DÉBUT DU TEST POUR : Gabriel Attal
🚦 PHRASE ACTUELLE : 'La dette de la sécurité sociale est de 30 milliards. J'aime ...'
✨ TEXTE NETTOYÉ : 'La dette de la sécurité sociale est de 30 milliards'
🚀 EXÉCUTION : 1 agent(s) en parallèle...
📊 [Agent Stat] Investigation approfondie en cours...
📝 [Rédacteur en Chef] Formatage strict pour affichage TV...

⏱️ Temps total du Pipeline : 11.76s

--- BANDEAU OBS FINAL ---
{
  "afficher_bandeau": true,
  "verdict_global": "Faux",
  "explications": {
    "statistique": {
      "texte": "FAUX : La réalité est de 163,3 milliards.",
      "url": "https://www.senat.fr/rap/a25-126/a25-1263.html",
      "source": ""
    }
  },
  "sources": [
    {
      "organization": "senat.fr",
      "url": "https://www.senat.fr/rap/a25-126/a25-1263.html"
    },
    {
      "organization": "economie.gouv.fr",
      "url": "https://www.economie.gouv.fr/daj/lettre-de-la-daj-deficit-de-la-securite-sociale-la-cour-des-comptes-formule-ses-recommandations"
    },
    {
      

In [9]:
dataset_pipeline_expert = [
    # --- 5 EXEMPLES AVEC QUESTIONS (Q&A) ---
    {
        "p": "Gabriel Attal", "q": "Que faites-vous pour les sans-abris ?",
        "ctx": "La dignité humaine est au cœur de notre action. Nous avons ouvert 200 000 places d'hébergement d'urgence. L'État mobilise des moyens sans précédent cet hiver. Les préfets ont reçu des instructions claires pour ne laisser personne dehors. Nous avons investi 2 milliards d'euros dans le plan Logement d'Abord. Les associations sont nos partenaires quotidiens. La solidarité nationale doit jouer à plein. Nous ne laissons personne sur le côté. Le combat contre la pauvreté est une priorité.",
        "a": "Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui."
    },
    {
        "p": "Jordan Bardella", "q": "Quelle est votre position sur l'énergie ?",
        "ctx": "La France doit retrouver sa puissance électrique. Le marché européen nous impose des prix délirants qui ruinent les familles. Nous avons un parc nucléaire exceptionnel mais il a été saboté par l'idéologie. Il faut arrêter de fermer des centrales. La souveraineté énergétique est la base de l'indépendance nationale. Nos entreprises délocalisent à cause du coût de l'énergie. Le gaz russe a été remplacé par du gaz de schiste américain plus cher. C'est un non-sens écologique et économique.",
        "a": "Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%."
    },
    {
        "p": "Marine Tondelier", "q": "Faut-il interdire les pesticides ?",
        "ctx": "La santé des Français n'est pas négociable. L'eau que nous buvons est polluée par des molécules chimiques. Les agriculteurs sont les premières victimes de ces produits. On observe une chute dramatique de la biodiversité dans nos campagnes. Les abeilles disparaissent à un rythme alarmant. Le gouvernement recule sans cesse face aux lobbies industriels. Il faut accompagner la transition au lieu de la punir. L'agroécologie est la seule voie de survie pour nos sols.",
        "a": "L'utilisation du glyphosate a augmenté de 50% en France cette année."
    },
    {
        "p": "Bruno Le Maire", "q": "Où en sont les impôts ?",
        "ctx": "Nous avons fait le choix de la politique de l'offre. Baisser les impôts des entreprises permet de créer des emplois. Nous avons supprimé la taxe d'habitation pour tous les Français. La pression fiscale était trop forte dans ce pays. Nous voulons récompenser le travail et l'effort. Les prélèvements obligatoires ont commencé à refluer. C'est une trajectoire de long terme que nous tenons. La croissance revient grâce à cette stabilité fiscale.",
        "a": "Nous avons réussi à diviser par deux la dette de la France en cinq ans."
    },
    {
        "p": "Sandrine Rousseau", "q": "Pourquoi cibler les jets privés ?",
        "ctx": "On demande des efforts de sobriété aux Français modestes. On leur dit de baisser le chauffage à 19 degrés. Mais les ultra-riches continuent de brûler la planète en toute impunité. Un trajet en jet émet plus de CO2 qu'un Français moyen en un an. C'est une injustice climatique insupportable. Le gouvernement refuse de réguler ce secteur. L'écologie ne peut pas être punitive pour les pauvres et permissive pour les riches.",
        "a": "Mais la vraie question, c'est de savoir si le gouvernement va taxer les superprofits de Total !"
    },
    # --- 10 EXEMPLES DISCOURS BRUTS (SANS QUESTIONS) ---
    {
        "p": "Emmanuel Macron", "q": "",
        "ctx": "Nous vivons une période de grandes transformations mondiales. La France doit être aux avant-postes de la technologie. Nous investissons massivement dans l'IA et le quantique. L'école est le berceau de cette ambition future. Il faut réarmer notre éducation nationale. Le niveau en mathématiques doit devenir une priorité dès le CP. Nous avons déjà commencé à recruter davantage de professeurs. La formation continue sera le pilier de notre réforme.",
        "a": "Nous avons recruté 100 000 policiers supplémentaires cette année."
    },
    {
        "p": "Jean-Luc Mélenchon", "q": "",
        "ctx": "Le peuple a faim pendant que les riches se gavent. Les prix à la consommation étranglent les familles. On nous parle de croissance, mais la croissance de quoi ? De la misère ! Nous proposons le blocage des prix des produits de première nécessité. Il faut augmenter le SMIC immédiatement. La vie chère n'est pas une fatalité, c'est un choix politique. Les banques centrales ne font qu'aggraver la situation en montant les taux.",
        "a": "Le SMIC en France est actuellement à 2500 euros net."
    },
    {
        "p": "Marine Le Pen", "q": "",
        "ctx": "La sécurité est la première des libertés. Nos villes et nos villages sont livrés à une délinquance de plus en plus barbare. Le laxisme judiciaire encourage les récidivistes. Il faut rétablir l'autorité partout. Les forces de l'ordre doivent être soutenues par l'État. Nous avons besoin de 85 000 places de prison supplémentaires. Le lien entre immigration et insécurité n'est plus à démontrer pour personne. Les Français veulent vivre en paix.",
        "a": "Le taux de cambriolages a baissé de 40% en un an sous ce gouvernement."
    },
    {
        "p": "Rachida Dati", "q": "",
        "ctx": "La culture est le ciment de notre Nation. Elle ne doit pas être réservée aux élites parisiennes. Chaque village doit avoir accès à une programmation de qualité. Nous soutenons le patrimoine religieux et rural. C'est l'âme de nos territoires. Le pass culture est un outil formidable de démocratisation. Nous allons renforcer les moyens des bibliothèques de quartier. L'art doit entrer dans les écoles dès le plus jeune âge.",
        "a": "Le budget de la culture est le premier poste de dépense de l'État."
    },
    {
        "p": "Olivier Faure", "q": "",
        "ctx": "La réforme des retraites est une blessure sociale qui ne se refermera pas. Travailler jusqu'à 64 ans est une injustice pour ceux qui ont des carrières longues. Les femmes sont les grandes perdantes de cette loi. Nous proposons le retour à la retraite à 60 ans. La pénibilité doit être réellement prise en compte. On ne peut pas traiter un ouvrier comme un cadre. Le financement est possible en taxant les revenus du capital.",
        "a": "La réforme des retraites a été votée à l'unanimité par le Parlement."
    },
    {
        "p": "Eric Ciotti", "q": "",
        "ctx": "La France subit un dérapage budgétaire incontrôlé. Le déficit public atteint des sommets alarmants. Le gouvernement dépense l'argent qu'il n'a pas. Nos enfants hériteront d'une dette colossale. Il faut sabrer dans la dépense publique inutile. L'État doit se concentrer sur ses missions régaliennes. Nous sommes les champions du monde des prélèvements obligatoires. Il faut libérer les entreprises de ce poids mort administratif.",
        "a": "Le déficit public de la France est tombé à 1% du PIB en 2025."
    },
    {
        "p": "Manuel Bompard", "q": "",
        "ctx": "La Ve République est une monarchie présidentielle. Un seul homme décide de tout pour 67 millions de citoyens. L'article 49.3 est devenu l'outil de la brutalité politique. Nous devons passer à une VIe République parlementaire. Le référendum d'initiative citoyenne doit être instauré. Il faut redonner du pouvoir aux élus locaux. La démocratie ne peut pas se résumer à un vote tous les cinq ans. Le peuple doit pouvoir révoquer ses élus.",
        "a": "L'article 49.3 a été utilisé 100 fois cette année."
    },
    {
        "p": "Gérald Darmanin", "q": "",
        "ctx": "La lutte contre les stupéfiants est notre combat quotidien. Les opérations 'place nette' déstabilisent les réseaux. Nous saisissons des quantités records de cocaïne et de cannabis. Les points de deal reculent dans nos quartiers populaires. Nous protégeons les plus jeunes de cette économie souterraine. Les moyens de la gendarmerie sont renforcés. Nous créons 200 nouvelles brigades sur tout le territoire. L'ordre est la condition du progrès social.",
        "a": "Il y a plus de policiers à Paris qu'à New York."
    },
    {
        "p": "Yannick Jadot", "q": "",
        "ctx": "Le climat ne nous attend pas. Les rapports du GIEC sont de plus en plus alarmants. Nous devons sortir des énergies fossiles le plus vite possible. Les transports représentent un tiers de nos émissions. Le train doit devenir moins cher que l'avion. Nous devons isoler tous les bâtiments de ce pays. C'est bon pour la planète et pour le porte-monnaie. L'écologie est une opportunité industrielle majeure pour la France. La nature n'est pas une ressource infinie.",
        "a": "La France est le pays le plus polluant au monde par habitant."
    },
    {
        "p": "François Ruffin", "q": "",
        "ctx": "Le travail doit payer. Aujourd'hui, on peut travailler à temps plein et vivre sous le seuil de pauvreté. C'est indigne. La précarité explose dans les métiers du soin et du lien. Les aides à domicile sont les oubliées de la République. Elles font des kilomètres sans être remboursées. Il faut un salaire minimum européen. Les profits records ne ruissellent pas vers ceux qui produisent. La dignité ouvrière doit être remise au centre.",
        "a": "Le nombre de milliardaires en France a baissé de 50% sous ce mandat."
    }
]

In [10]:
import time
import json
import asyncio

async def run_benchmark_tv_production():
    print(f"\n⏱️  Lancement du test d'affichage OBS ({len(dataset_pipeline_expert)} tests)...")
    print("="*80)

    for i, item in enumerate(dataset_pipeline_expert):
        try:
            start_time = time.perf_counter()
            
            # Préparation des données pour ce test
            data_pour_agents = {
                "personne": item['p'],
                "question_posee": item.get('q', ''),
                "affirmation": item['a'],
                "contexte_precedent": item.get('ctx', '')
            }

            print(f"\n🎬 TEST {i+1}/{len(dataset_pipeline_expert)} | Sujet : {item['p']}")
            print(f"🗣️  PHRASE : '{item['a']}'")
            
            # --- 1. ANALYSE PARALLÈLE ---
            rapports_bruts = await executer_analyse_parallele(data_pour_agents)
            
            # --- 2. ARBITRAGE DU JUGE ---
            juge_output = await agent_editeur(
                contexte_precedent=data_pour_agents["contexte_precedent"], 
                affirmation_actuelle=data_pour_agents["affirmation"], 
                rapports_agents=rapports_bruts
            )
            
            # --- 3. ENRICHISSEMENT DES SOURCES ---
            if isinstance(juge_output, dict):
                bandeau_tv_final = _enrich_editor_result_with_sources(juge_output, rapports_bruts)
            else:
                bandeau_tv_final = {"afficher_bandeau": False, "raison": "Erreur format Juge."}

            # --- 4. LOGIQUE D'AFFICHAGE OBS ---
            afficher = bandeau_tv_final.get('afficher_bandeau', False)
            verdict = bandeau_tv_final.get('verdict_global', 'Indéterminé')
            
            if afficher:
                explications = bandeau_tv_final.get('explications', {})
                texte_obs = ""
                source_nom = ""

                # 🏆 SYSTÈME DE PRIORITÉ D'AFFICHAGE (1 seul bandeau !)
                if "statistique" in explications and explications["statistique"].get("texte"):
                    texte_obs = explications["statistique"]["texte"]
                    source_nom = explications["statistique"].get("source", "Source officielle")
                elif "contexte" in explications and explications["contexte"].get("texte"):
                    texte_obs = explications["contexte"]["texte"]
                    source_nom = explications["contexte"].get("source", "Source officielle")
                elif "coherence" in explications and explications["coherence"].get("texte"):
                    texte_obs = explications["coherence"]["texte"]
                    source_nom = explications["coherence"].get("source", "Source officielle")
                elif "rhetorique" in explications and explications["rhetorique"]:
                    texte_obs = explications["rhetorique"]
                    source_nom = "Analyse"

                if texte_obs:
                    print(f"🚨 [ENVOI OBS] -> {texte_obs} (Source : {source_nom})")
                else:
                    print("🔇 [SILENCE OBS] Erreur de formatage du Juge.")
            else:
                raison = bandeau_tv_final.get('raison', '')
                print(f"🔇 [SILENCE OBS] (Verdict: {verdict}) {raison}")

            end_time = time.perf_counter()
            print(f"⏱️  Temps : {end_time - start_time:.2f}s")

        except Exception as e:
            # 💡 LE BLOC MANQUANT : Il attrape l'erreur et affiche ce qui ne va pas sans stopper tout le benchmark
            print(f"❌ Erreur critique au test {i+1} : {e}")

        # 💤 Pause obligatoire après chaque test (réussi ou raté) pour le Rate Limit
        print("-" * 80)
        await asyncio.sleep(10)

# Lancement de la boucle
await run_benchmark_tv_production()


⏱️  Lancement du test d'affichage OBS (15 tests)...

🎬 TEST 1/15 | Sujet : Gabriel Attal
🗣️  PHRASE : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
🚦 PHRASE ACTUELLE : 'Grâce à notre action, il n'y a plus aucun SDF en France aujo...'
✨ TEXTE NETTOYÉ : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
🚀 EXÉCUTION : 1 agent(s) en parallèle...
📊 [Agent Stat] Investigation approfondie en cours...
📝 [Rédacteur en Chef] Formatage strict pour affichage TV...
🚨 [ENVOI OBS] -> FAUX : La réalité est de 295 000 à 315 000. (Source : immobilier-mohammedia.com)
⏱️  Temps : 6.96s
--------------------------------------------------------------------------------

🎬 TEST 2/15 | Sujet : Jordan Bardella
🗣️  PHRASE : 'Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%.'
🚦 PHRASE ACTUELLE : 'Aujourd'hui, le nucléaire ne représente plus que 20% de notr...'
✨ TEXTE NETTOYÉ : 'Aujourd'hui, le nucléaire ne représente plus que 65%